In [0]:
import requests
from pyspark.sql.functions import current_timestamp

In [0]:
dbutils.widgets.text("API_KEY", "")
dbutils.widgets.text("SUBGRAPH_ID", "")

In [0]:
API_KEY = dbutils.widgets.get("API_KEY")
SUBGRAPH_ID = dbutils.widgets.get("SUBGRAPH_ID")

In [0]:
SUBGRAPH_UNIV3_URL = f"https://gateway.thegraph.com/api/{API_KEY}/subgraphs/id/{SUBGRAPH_ID}"
query_variables = {
    "skip": 1000
}

In [0]:
pool_query = """
query PoolsQuery($skip: Int!){
  pools(first: 1000,
    skip: $skip, 
    orderDirection: asc
  ) {
    id
    token0 { symbol id decimals }
    token1 { symbol id decimals }
    feeTier
    liquidity
    sqrtPrice
    tick
    createdAtTimestamp
    createdAtBlockNumber
  }
}
"""

In [0]:
response = requests.post(SUBGRAPH_UNIV3_URL, json={"query": pool_query, "variables": query_variables})

In [0]:
df = spark.createDataFrame(response.json()["data"]["pools"])

df = df.withColumn("_load_ts", current_timestamp())

In [0]:
display(df)

In [0]:
df.write.format("delta").mode("append").saveAsTable("uniswap.bronze.pools")